# GNN Training Notebook
**Phase 3 — F1 Deep Learning Project**

Models the F1 race as a graph where each driver is a node and edges connect drivers close to each other on track.

In [1]:
# Cell 1 — Imports
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import r2_score, mean_absolute_error
import matplotlib.pyplot as plt
import sys
sys.path.append('..')
from models.gnn_model import RaceGNN, build_adjacency
print('All imports done!')

All imports done!


In [2]:
# Cell 2 — Load Data and Build Race Graphs
df = pd.read_csv('../f1.csv', index_col=0).dropna()

features = ['lap','position','pit_stop','tyre_age','grid',
            'alt','driver_skill','circuit_difficulty',
            'race_year','round','prev_lap_time']

# Build one graph per (raceId, lap) combination
graphs = []
for (race_id, lap_num), grp in df.groupby(['raceId','lap']):
    if len(grp) < 3:
        continue
    node_feats = torch.FloatTensor(grp[features].values)
    positions  = grp['position'].tolist()
    adj        = build_adjacency(positions, threshold=5)
    targets    = torch.FloatTensor(grp['lap_time_ms'].values)
    graphs.append((node_feats, adj, targets))

print(f'Total race graphs: {len(graphs)}')

C:\Users\Asus'\AppData\Local\Temp\ipykernel_19580\2341892973.py:2: DtypeWarning: Columns (0: positionOrder) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../f1.csv', index_col=0).dropna()


KeyError: "['driver_skill', 'circuit_difficulty', 'prev_lap_time'] not in index"

In [ ]:
# Cell 3 — Train GNN
model     = RaceGNN(in_features=11, hidden=64, num_layers=3)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn   = nn.MSELoss()

split        = int(0.8 * len(graphs))
train_graphs = graphs[:split]
test_graphs  = graphs[split:]

train_losses = []
EPOCHS = 20

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    for node_feats, adj, targets in train_graphs:
        preds      = model(node_feats, adj)
        loss       = loss_fn(preds, targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    avg = epoch_loss / len(train_graphs)
    train_losses.append(avg)
    print(f'Epoch {epoch+1}/{EPOCHS} | Loss: {avg:.2f}')

import torch
torch.save(model.state_dict(), '../gnn_model.pth')
print('GNN model saved!')

In [ ]:
# Cell 4 — Evaluate GNN
model.eval()
all_preds, all_targets = [], []

with torch.no_grad():
    for node_feats, adj, targets in test_graphs:
        preds = model(node_feats, adj)
        all_preds.extend(preds.numpy())
        all_targets.extend(targets.numpy())

r2  = r2_score(all_targets, all_preds)
mae = mean_absolute_error(all_targets, all_preds) / 1000
print(f'GNN R2  : {r2:.4f}')
print(f'GNN MAE : {mae:.3f} seconds')

In [ ]:
# Cell 5 — Plot Training Loss
plt.figure(figsize=(10,5))
plt.plot(train_losses, color='blue', linewidth=2)
plt.title('GNN Training Loss Over Epochs', fontsize=14)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.grid(True)
plt.tight_layout()
plt.savefig('../gnn_training_loss.png')
plt.show()
print('Plot saved!')